# Exploring Raw Helius API Data
This notebook shows how raw data flows through the pipeline

In [ ]:
# Cell 1: Setup
import httpx
import json
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('HELIUS_API_KEY')
print(f"API Key loaded: {api_key[:8]}..." if api_key else "No API key found!")

In [ ]:
# Cell 2: Fetch raw data from Helius API (for USDC)
usdc_mint = 'EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v'

# In Jupyter, you can use httpx synchronously
with httpx.Client(timeout=30.0) as client:
    url = f'https://api.helius.xyz/v0/addresses/{usdc_mint}/transactions'
    response = client.get(url, params={'api-key': api_key, 'limit': 1})
    raw_data = response.json()

print("Raw Helius API Response:")
print(json.dumps(raw_data[0], indent=2))

In [ ]:
# Cell 3: Extract key fields from the raw response
tx = raw_data[0]

print("=== Transaction Summary ===")
print(f"Signature: {tx['signature']}")
print(f"Type: {tx['type']}")
print(f"Source/Protocol: {tx['source']}")
print(f"Fee: {tx['fee']} lamports ({tx['fee']/1e9:.6f} SOL)")
print(f"Fee Payer: {tx['feePayer']}")
print()
print("=== Token Transfers ===")
for i, transfer in enumerate(tx.get('tokenTransfers', [])):
    mint = transfer['mint']
    symbol = 'USDC' if 'EPjFWdd5' in mint else 'PYUSD' if '2b1kV6' in mint else mint[:8]
    print(f"[{i+1}] {transfer['tokenAmount']:,.6f} {symbol}")
    print(f"    From: {transfer['fromUserAccount']}")
    print(f"    To: {transfer['toUserAccount']}")

In [ ]:
# Cell 4: View data in DuckDB Bronze layer (after ingestion)
import duckdb

db = duckdb.connect('solana_stablecoins.duckdb', read_only=True)

print("=== BRONZE LAYER (Raw from Helius, normalized by dlt) ===")
print()
print("Tables in bronze schema:")
tables = db.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'bronze'").fetchall()
for t in tables:
    print(f"  - bronze.{t[0]}")

In [ ]:
# Cell 5: View raw transactions
df = db.execute("SELECT * FROM bronze.transactions LIMIT 5").fetchdf()
df

In [ ]:
# Cell 6: View token transfers (nested data normalized by dlt)
df = db.execute("SELECT * FROM bronze.transactions__token_transfers LIMIT 5").fetchdf()
df

In [ ]:
# Cell 7: View Silver layer (decoded stablecoin transfers)
print("=== SILVER LAYER (Decoded & filtered) ===")
df = db.execute("SELECT * FROM silver.stablecoin_transfers LIMIT 10").fetchdf()
df

In [ ]:
# Cell 8: View Gold layer (analytics)
print("=== GOLD LAYER (Analytics) ===")
print()
print("Daily Flows:")
db.execute("SELECT * FROM gold.daily_stablecoin_flows").fetchdf()

In [ ]:
# Cell 9: Wallet Classifications
print("Top classified wallets:")
db.execute("""
    SELECT wallet, wallet_label, wallet_category, total_volume, confidence
    FROM gold.wallet_analytics 
    WHERE wallet_category != 'unknown'
    ORDER BY total_volume DESC
    LIMIT 15
""").fetchdf()

In [ ]:
# Cell 10: DEX Flow breakdown
print("Volume by Protocol:")
db.execute("""
    SELECT protocol, SUM(volume) as total_volume, SUM(tx_count) as total_txs
    FROM gold.dex_flows
    GROUP BY protocol
    ORDER BY total_volume DESC
    LIMIT 10
""").fetchdf()